# Chapter 7.5: Fused Kernels — RMSNorm, SiLU-Mul, Rotary Embedding

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Explain why kernel fusion matters** for LLM serving performance
2. **Implement fused RMSNorm** in both CUDA and Triton
3. **Implement fused SiLU-Mul** (gated activation) kernel
4. **Implement Rotary Positional Embedding (RoPE)** kernel
5. **Fuse residual add + normalization** into a single kernel
6. **Profile and measure** memory bandwidth savings from fusion

## Prerequisites

- Chapter 7.1 (CUDA Primer) and 7.2 (Triton Programming)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part7/chapter_7.5_fused_kernels.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part7/chapter_7.5_fused_kernels.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Why Kernel Fusion Matters

### 1.1 The Memory Wall

Modern GPUs have a massive gap between compute and memory bandwidth:

| GPU | FP16 Compute | HBM Bandwidth | Ratio |
|-----|-------------|---------------|-------|
| A100 | 312 TFLOPS | 2.0 TB/s | 156 FLOPS/byte |
| H100 | 990 TFLOPS | 3.4 TB/s | 291 FLOPS/byte |

Most LLM operations (normalization, activation, position encoding) perform only **1-10 FLOPs per byte loaded**. They are severely **memory-bound**.

### 1.2 The Cost of Separate Kernels

Consider `y = RMSNorm(x + residual)` as two separate kernels:

```
Kernel 1: tmp = x + residual     → Read x, residual (2 reads), Write tmp (1 write)
Kernel 2: y = RMSNorm(tmp)       → Read tmp (1 read), Write y (1 write)
Total: 3 reads + 2 writes = 5 memory operations
```

Fused into one kernel:
```
Kernel 1: y = RMSNorm(x + residual)  → Read x, residual (2 reads), Write y (1 write)
Total: 2 reads + 1 write = 3 memory operations
```

**40% less memory traffic!** For a memory-bound kernel, this translates directly to ~40% speedup.

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport numpy as np# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))fig.suptitle('Kernel Fusion: Reducing Memory Traffic', fontsize=14,             fontweight='bold', color='#2C3E50')# Unfusedax1.set_xlim(0, 10)ax1.set_ylim(0, 7)ax1.axis('off')ax1.set_title('Unfused (3 Kernel Launches)', fontsize=11, fontweight='bold', color=RED, pad=10)ops_unfused = [    (5, 5.5, "HBM", GRAY, 'Load x'),    (2, 4, "RMSNorm", BLUE, None),    (5, 4, "HBM", GRAY, 'Store tmp'),    (5, 3, "HBM", GRAY, 'Load tmp'),    (8, 4, "SiLU", GREEN, None),    (5, 2, "HBM", GRAY, 'Store tmp2'),    (5, 1, "HBM", GRAY, 'Load tmp2'),    (5, 0.5, "Multiply", ORANGE, 'Store result'),]for i, (x, y, label, color, note) in enumerate(ops_unfused):    if "HBM" in label:        ax1.annotate('', xy=(x, y-0.2), xytext=(x, y+0.2),                    arrowprops=dict(arrowstyle='<->' if note and 'Load' in note else '->',                                   color=RED, lw=2))        if note:            ax1.text(x+0.3, y, note, fontsize=7, color=RED, va='center')    else:        box = mpatches.FancyBboxPatch((x-0.8, y-0.3), 1.6, 0.6, boxstyle="round,pad=0.05",                                       facecolor=color, alpha=0.85, edgecolor='white', lw=1.5)        ax1.add_patch(box)        ax1.text(x, y, label, ha='center', va='center', fontsize=9, color='white', fontweight='bold')ax1.text(5, -0.3, '6 memory operations', fontsize=10, ha='center', color=RED, fontweight='bold')# Fusedax2.set_xlim(0, 10)ax2.set_ylim(0, 7)ax2.axis('off')ax2.set_title('Fused (1 Kernel Launch)', fontsize=11, fontweight='bold', color=GREEN, pad=10)# Single fused kernelax2.annotate('', xy=(5, 5.2), xytext=(5, 5.8),            arrowprops=dict(arrowstyle='->', color=GREEN, lw=2.5))ax2.text(5.3, 5.5, 'Load x', fontsize=8, color=GREEN, va='center')fused_box = mpatches.FancyBboxPatch((2.5, 2.5), 5, 2.5, boxstyle="round,pad=0.15",                                     facecolor=GREEN, alpha=0.85, edgecolor='white', lw=2)ax2.add_patch(fused_box)ax2.text(5, 4.2, 'Fused Kernel', fontsize=12, fontweight='bold', color='white', ha='center')ax2.text(5, 3.5, 'RMSNorm + SiLU + Multiply', fontsize=9, color='white', ha='center')ax2.text(5, 2.9, '(all in registers/SRAM)', fontsize=8, color='white', ha='center', style='italic')ax2.annotate('', xy=(5, 1.8), xytext=(5, 2.3),            arrowprops=dict(arrowstyle='->', color=GREEN, lw=2.5))ax2.text(5.3, 2.0, 'Store result', fontsize=8, color=GREEN, va='center')ax2.text(5, 0.8, '2 memory operations\n(67% reduction!)', fontsize=10, ha='center',         color=GREEN, fontweight='bold',         bbox=dict(boxstyle='round,pad=0.3', facecolor=LIGHT_GREEN, alpha=0.3))plt.tight_layout(rect=[0, 0, 1, 0.93])plt.savefig("kernel_fusion.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
# Quantify the fusion benefit
def analyze_fusion_benefit(hidden_size=4096, batch_size=256, dtype_bytes=2):
    """
    Analyze memory traffic savings from kernel fusion in a typical LLM layer.
    """
    elements = hidden_size * batch_size
    bytes_per_op = elements * dtype_bytes
    
    print(f"Configuration: hidden_size={hidden_size}, batch_size={batch_size}, FP{dtype_bytes*8}")
    print(f"Elements per tensor: {elements:,} ({bytes_per_op/1e6:.1f} MB)")
    print("=" * 70)
    
    operations = [
        {
            "name": "Residual + RMSNorm",
            "unfused_reads": 4,  # x, residual (kernel1) + tmp (kernel2)
            "unfused_writes": 2,  # tmp (kernel1) + y (kernel2)
            "fused_reads": 2,     # x, residual
            "fused_writes": 1,    # y
        },
        {
            "name": "SiLU + Multiply (gate)",
            "unfused_reads": 3,  # x (silu) + silu_out, gate (multiply)
            "unfused_writes": 2,  # silu_out + result
            "fused_reads": 2,     # x, gate
            "fused_writes": 1,    # result
        },
        {
            "name": "RoPE (sin/cos + multiply)",
            "unfused_reads": 4,  # x, cos, sin (for rotation) + intermediate
            "unfused_writes": 2,
            "fused_reads": 3,     # x, cos, sin
            "fused_writes": 1,    # result
        },
    ]
    
    total_unfused = 0
    total_fused = 0
    
    print(f"{'Operation':<30} {'Unfused IO':>12} {'Fused IO':>12} {'Savings':>10}")
    print("-" * 65)
    
    for op in operations:
        unfused = (op['unfused_reads'] + op['unfused_writes']) * bytes_per_op
        fused = (op['fused_reads'] + op['fused_writes']) * bytes_per_op
        savings = 1 - fused / unfused
        total_unfused += unfused
        total_fused += fused
        print(f"{op['name']:<30} {unfused/1e6:>10.1f} MB {fused/1e6:>10.1f} MB {savings:>9.0%}")
    
    print("-" * 65)
    print(f"{'TOTAL':<30} {total_unfused/1e6:>10.1f} MB {total_fused/1e6:>10.1f} MB {1-total_fused/total_unfused:>9.0%}")
    print(f"\nOn A100 (2 TB/s): unfused={total_unfused/2e12*1e6:.1f}µs, fused={total_fused/2e12*1e6:.1f}µs")

analyze_fusion_benefit()

## 2. Fused RMSNorm

### 2.1 Algorithm

RMSNorm (Root Mean Square Normalization) is used in LLaMA, Gemma, and most modern LLMs:

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}} \cdot \gamma$$

Where:
- $x \in \mathbb{R}^d$ is the input vector (one row)
- $\gamma \in \mathbb{R}^d$ is the learnable weight
- $\epsilon$ is a small constant for numerical stability

### 2.2 Why RMSNorm Instead of LayerNorm?

RMSNorm is simpler than LayerNorm:
- **No mean subtraction** (saves one reduction)
- **No bias term** (fewer parameters)
- Empirically works just as well for LLMs

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def rmsnorm_fwd_kernel(
    input_ptr,    # (num_tokens, hidden_size)
    weight_ptr,   # (hidden_size,)
    output_ptr,   # (num_tokens, hidden_size)
    hidden_size,
    eps,
    stride_in,    # Stride between rows of input
    stride_out,   # Stride between rows of output
    BLOCK_SIZE: tl.constexpr,
):
    """
    Fused RMSNorm kernel.
    
    Each program processes one row (one token).
    Steps:
      1. Load the row
      2. Compute sum of squares
      3. Compute 1/sqrt(mean_sq + eps)
      4. Multiply by weight and store
    
    All in one pass — only 1 read of input, 1 write of output!
    """
    row_idx = tl.program_id(0)
    
    # Offsets within the row
    col_offsets = tl.arange(0, BLOCK_SIZE)
    mask = col_offsets < hidden_size
    
    # Load input row (cast to float32 for precision)
    row_ptr = input_ptr + row_idx * stride_in
    x = tl.load(row_ptr + col_offsets, mask=mask, other=0.0).to(tl.float32)
    
    # Compute RMS: sqrt(mean(x^2) + eps)
    x_sq = x * x
    mean_sq = tl.sum(x_sq, axis=0) / hidden_size
    rrms = tl.rsqrt(mean_sq + eps)  # 1 / sqrt(mean_sq + eps)
    
    # Normalize and apply weight
    x_norm = x * rrms
    weight = tl.load(weight_ptr + col_offsets, mask=mask)
    result = x_norm * weight
    
    # Store result
    out_ptr = output_ptr + row_idx * stride_out
    tl.store(out_ptr + col_offsets, result, mask=mask)


def rmsnorm_triton(x, weight, eps=1e-6):
    """RMSNorm using Triton kernel."""
    num_tokens, hidden_size = x.shape
    output = torch.empty_like(x)
    BLOCK_SIZE = triton.next_power_of_2(hidden_size)
    
    rmsnorm_fwd_kernel[(num_tokens,)](
        x, weight, output,
        hidden_size, eps,
        x.stride(0), output.stride(0),
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return output


# Reference implementation
def rmsnorm_reference(x, weight, eps=1e-6):
    rms = torch.sqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + eps)
    return (x.float() / rms * weight).to(x.dtype)


if torch.cuda.is_available():
    hidden_size = 4096
    num_tokens = 256
    
    x = torch.randn(num_tokens, hidden_size, device='cuda', dtype=torch.float32)
    weight = torch.ones(hidden_size, device='cuda', dtype=torch.float32)
    
    y_triton = rmsnorm_triton(x, weight)
    y_ref = rmsnorm_reference(x, weight)
    
    print(f"RMSNorm ({num_tokens} tokens, hidden={hidden_size})")
    print(f"Max error: {(y_triton - y_ref).abs().max().item():.2e}")
    print(f"PASSED: {torch.allclose(y_triton, y_ref, atol=1e-4)}")

In [ ]:
# Fused Residual + RMSNorm

@triton.jit
def fused_residual_rmsnorm_kernel(
    input_ptr,
    residual_ptr,
    weight_ptr,
    output_ptr,
    residual_out_ptr,  # Updated residual (input + residual)
    hidden_size,
    eps,
    stride_in, stride_res, stride_out, stride_res_out,
    BLOCK_SIZE: tl.constexpr,
):
    """
    Fused residual connection + RMSNorm.
    
    Computes:
      residual_out = input + residual
      output = RMSNorm(residual_out) * weight
    
    In one kernel! Saves one full read+write of the intermediate.
    """
    row_idx = tl.program_id(0)
    col_offsets = tl.arange(0, BLOCK_SIZE)
    mask = col_offsets < hidden_size
    
    # Load input and residual
    x = tl.load(input_ptr + row_idx * stride_in + col_offsets, mask=mask, other=0.0).to(tl.float32)
    res = tl.load(residual_ptr + row_idx * stride_res + col_offsets, mask=mask, other=0.0).to(tl.float32)
    
    # Residual connection
    hidden = x + res
    
    # Store updated residual (needed for next layer's residual connection)
    tl.store(residual_out_ptr + row_idx * stride_res_out + col_offsets, hidden, mask=mask)
    
    # RMSNorm
    mean_sq = tl.sum(hidden * hidden, axis=0) / hidden_size
    rrms = tl.rsqrt(mean_sq + eps)
    
    weight = tl.load(weight_ptr + col_offsets, mask=mask)
    result = hidden * rrms * weight
    
    tl.store(output_ptr + row_idx * stride_out + col_offsets, result, mask=mask)


def fused_residual_rmsnorm(x, residual, weight, eps=1e-6):
    num_tokens, hidden_size = x.shape
    output = torch.empty_like(x)
    residual_out = torch.empty_like(x)
    BLOCK_SIZE = triton.next_power_of_2(hidden_size)
    
    fused_residual_rmsnorm_kernel[(num_tokens,)](
        x, residual, weight, output, residual_out,
        hidden_size, eps,
        x.stride(0), residual.stride(0), output.stride(0), residual_out.stride(0),
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return output, residual_out


if torch.cuda.is_available():
    x = torch.randn(256, 4096, device='cuda')
    residual = torch.randn(256, 4096, device='cuda')
    weight = torch.ones(4096, device='cuda')
    
    out, res_out = fused_residual_rmsnorm(x, residual, weight)
    
    # Reference
    ref_res = x + residual
    ref_out = rmsnorm_reference(ref_res, weight)
    
    print(f"Fused Residual + RMSNorm")
    print(f"  Residual error: {(res_out - ref_res).abs().max().item():.2e}")
    print(f"  Output error:   {(out - ref_out).abs().max().item():.2e}")
    print(f"  PASSED: {torch.allclose(out, ref_out, atol=1e-4)}")

## 3. Fused SiLU-Mul (Gated Activation)

### 3.1 The SiLU (Swish) Activation

$$\text{SiLU}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

### 3.2 Gated Linear Unit (GLU) Variant

In LLaMA/Mistral FFN layers, the operation is:

$$\text{output} = \text{SiLU}(W_{gate} \cdot x) \odot (W_{up} \cdot x)$$

Where $\odot$ is element-wise multiplication. The gate and up projections often produce a concatenated tensor that we split:

```python
# Unfused
gate = linear_gate(x)    # (batch, hidden)
up = linear_up(x)        # (batch, hidden)
silu_out = F.silu(gate)  # Read gate, write silu_out
result = silu_out * up   # Read silu_out, up, write result

# Fused
result = fused_silu_mul(gate, up)  # Read gate, up, write result (skip silu_out!)
```

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def fused_silu_mul_kernel(
    gate_ptr,     # Input for SiLU activation
    up_ptr,       # Input for element-wise multiply
    output_ptr,   # Output = SiLU(gate) * up
    N,            # Total number of elements
    BLOCK_SIZE: tl.constexpr,
):
    """
    Fused SiLU activation + element-wise multiply.
    
    Computes: output = SiLU(gate) * up
            = gate * sigmoid(gate) * up
    
    This fuses two operations into one kernel:
    - Saves one write (SiLU intermediate) and one read (SiLU intermediate)
    - SiLU is computed in registers, never written to global memory
    """
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < N
    
    # Load both inputs
    gate = tl.load(gate_ptr + offsets, mask=mask).to(tl.float32)
    up = tl.load(up_ptr + offsets, mask=mask).to(tl.float32)
    
    # SiLU(gate) = gate * sigmoid(gate)
    # sigmoid(gate) = 1 / (1 + exp(-gate))
    sigmoid_gate = tl.sigmoid(gate)
    silu_gate = gate * sigmoid_gate
    
    # Fused multiply
    result = silu_gate * up
    
    # Single write
    tl.store(output_ptr + offsets, result, mask=mask)


def fused_silu_mul(gate, up):
    """Fused SiLU activation + multiply."""
    assert gate.shape == up.shape
    N = gate.numel()
    output = torch.empty_like(gate)
    BLOCK_SIZE = 1024
    grid = ((N + BLOCK_SIZE - 1) // BLOCK_SIZE,)
    
    fused_silu_mul_kernel[grid](gate, up, output, N, BLOCK_SIZE=BLOCK_SIZE)
    return output


if torch.cuda.is_available():
    batch, hidden = 256, 11008  # LLaMA-7B FFN hidden size
    
    gate = torch.randn(batch, hidden, device='cuda')
    up = torch.randn(batch, hidden, device='cuda')
    
    # Fused
    result_fused = fused_silu_mul(gate, up)
    
    # Reference
    result_ref = torch.nn.functional.silu(gate) * up
    
    error = (result_fused - result_ref).abs().max().item()
    print(f"Fused SiLU-Mul ({batch}×{hidden})")
    print(f"Max error: {error:.2e}")
    print(f"PASSED: {error < 1e-4}")

## 4. Rotary Positional Embedding (RoPE)

### 4.1 Complex Number Interpretation

RoPE encodes position by rotating pairs of dimensions in the query/key vectors. For position $m$ and dimension pair $(2i, 2i+1)$:

$$\begin{pmatrix} q'_{2i} \\ q'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}$$

Where $\theta_i = 10000^{-2i/d}$ is the frequency for the $i$-th dimension pair.

### 4.2 Equivalent Formulation

This can be computed more efficiently as:

$$q'_{2i} = q_{2i} \cos(m\theta_i) - q_{2i+1} \sin(m\theta_i)$$
$$q'_{2i+1} = q_{2i} \sin(m\theta_i) + q_{2i+1} \cos(m\theta_i)$$

### 4.3 Why Fuse RoPE?

Without fusion, RoPE requires:
1. Read q → compute rotation → write q_rotated
2. Read cos_cache, sin_cache

With fusion into the QKV projection or attention kernel, we avoid materializing the intermediate.

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def rope_kernel(
    q_ptr,          # Query tensor: (num_tokens, num_heads, head_dim)
    cos_ptr,        # Cosine cache: (max_seq_len, head_dim // 2)
    sin_ptr,        # Sine cache: (max_seq_len, head_dim // 2)
    positions_ptr,  # Position indices: (num_tokens,)
    output_ptr,     # Output: (num_tokens, num_heads, head_dim)
    num_heads,
    head_dim,
    # Strides for q: (num_tokens, num_heads, head_dim)
    stride_qt, stride_qh, stride_qd,
    # Strides for cos/sin: (max_seq_len, head_dim//2)
    stride_cs, stride_cd,
    # Strides for output
    stride_ot, stride_oh, stride_od,
    HALF_DIM: tl.constexpr,  # head_dim // 2
):
    """
    Rotary Positional Embedding (RoPE) kernel.
    
    Each program handles one (token, head) pair.
    Applies rotation to pairs of dimensions.
    """
    token_idx = tl.program_id(0)
    head_idx = tl.program_id(1)
    
    # Get position for this token
    position = tl.load(positions_ptr + token_idx)
    
    # Load cos and sin values for this position
    half_offsets = tl.arange(0, HALF_DIM)
    cos_vals = tl.load(cos_ptr + position * stride_cs + half_offsets * stride_cd)
    sin_vals = tl.load(sin_ptr + position * stride_cs + half_offsets * stride_cd)
    
    # Load query: even and odd dimensions
    q_base = q_ptr + token_idx * stride_qt + head_idx * stride_qh
    
    # Even dimensions: q[..., 0], q[..., 2], q[..., 4], ...
    even_offsets = half_offsets * 2
    q_even = tl.load(q_base + even_offsets * stride_qd)
    
    # Odd dimensions: q[..., 1], q[..., 3], q[..., 5], ...
    odd_offsets = half_offsets * 2 + 1
    q_odd = tl.load(q_base + odd_offsets * stride_qd)
    
    # Apply rotation
    # q_even' = q_even * cos - q_odd * sin
    # q_odd'  = q_even * sin + q_odd * cos
    out_even = q_even * cos_vals - q_odd * sin_vals
    out_odd = q_even * sin_vals + q_odd * cos_vals
    
    # Store rotated values
    out_base = output_ptr + token_idx * stride_ot + head_idx * stride_oh
    tl.store(out_base + even_offsets * stride_od, out_even)
    tl.store(out_base + odd_offsets * stride_od, out_odd)


def rope_triton(q, cos_cache, sin_cache, positions):
    """Apply RoPE to query tensor."""
    num_tokens, num_heads, head_dim = q.shape
    output = torch.empty_like(q)
    
    grid = (num_tokens, num_heads)
    
    rope_kernel[grid](
        q, cos_cache, sin_cache, positions, output,
        num_heads, head_dim,
        q.stride(0), q.stride(1), q.stride(2),
        cos_cache.stride(0), cos_cache.stride(1),
        output.stride(0), output.stride(1), output.stride(2),
        HALF_DIM=head_dim // 2,
    )
    return output


# Create cos/sin cache
def create_rope_cache(max_seq_len, head_dim, base=10000.0, device='cuda'):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    positions = torch.arange(max_seq_len, device=device).float()
    freqs = torch.outer(positions, inv_freq)  # (max_seq_len, head_dim//2)
    cos_cache = torch.cos(freqs)
    sin_cache = torch.sin(freqs)
    return cos_cache, sin_cache


if torch.cuda.is_available():
    num_tokens, num_heads, head_dim = 32, 8, 128
    max_seq_len = 2048
    
    q = torch.randn(num_tokens, num_heads, head_dim, device='cuda')
    cos_cache, sin_cache = create_rope_cache(max_seq_len, head_dim)
    positions = torch.arange(num_tokens, device='cuda', dtype=torch.int32)
    
    # Triton RoPE
    q_rotated = rope_triton(q, cos_cache, sin_cache, positions)
    
    # Reference RoPE
    cos_vals = cos_cache[positions]  # (num_tokens, head_dim//2)
    sin_vals = sin_cache[positions]
    q_even = q[..., 0::2]  # Even dimensions
    q_odd = q[..., 1::2]   # Odd dimensions
    ref_even = q_even * cos_vals[:, None, :] - q_odd * sin_vals[:, None, :]
    ref_odd = q_even * sin_vals[:, None, :] + q_odd * cos_vals[:, None, :]
    q_ref = torch.stack([ref_even, ref_odd], dim=-1).flatten(-2)
    
    error = (q_rotated - q_ref).abs().max().item()
    print(f"RoPE ({num_tokens} tokens, {num_heads} heads, dim={head_dim})")
    print(f"Max error: {error:.2e}")
    print(f"PASSED: {error < 1e-5}")

## 5. Performance Benchmarks: Fused vs Unfused

In [ ]:
import torch
import time

def benchmark_fn(fn, *args, warmup=20, n_iter=200):
    for _ in range(warmup):
        fn(*args)
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(n_iter):
        fn(*args)
    torch.cuda.synchronize()
    return (time.perf_counter() - start) / n_iter * 1e6  # microseconds


if torch.cuda.is_available():
    hidden_sizes = [2048, 4096, 8192]
    batch_size = 256
    
    print("=" * 80)
    print("BENCHMARK: Fused vs Unfused Operations")
    print("=" * 80)
    
    for hidden_size in hidden_sizes:
        print(f"\nHidden size: {hidden_size}, Batch: {batch_size}")
        print("-" * 60)
        
        x = torch.randn(batch_size, hidden_size, device='cuda')
        weight = torch.ones(hidden_size, device='cuda')
        residual = torch.randn(batch_size, hidden_size, device='cuda')
        gate = torch.randn(batch_size, hidden_size, device='cuda')
        up = torch.randn(batch_size, hidden_size, device='cuda')
        
        # RMSNorm
        unfused_rms = benchmark_fn(rmsnorm_reference, x, weight)
        fused_rms = benchmark_fn(rmsnorm_triton, x, weight)
        
        # Residual + RMSNorm
        def unfused_res_rmsnorm(x, res, w):
            tmp = x + res
            return rmsnorm_reference(tmp, w)
        
        unfused_res = benchmark_fn(unfused_res_rmsnorm, x, residual, weight)
        fused_res = benchmark_fn(fused_residual_rmsnorm, x, residual, weight)
        
        # SiLU-Mul
        def unfused_silu_mul(gate, up):
            return torch.nn.functional.silu(gate) * up
        
        unfused_silu = benchmark_fn(unfused_silu_mul, gate, up)
        fused_silu = benchmark_fn(fused_silu_mul, gate, up)
        
        print(f"  {'Operation':<30} {'Unfused (µs)':>12} {'Fused (µs)':>12} {'Speedup':>8}")
        print(f"  {'RMSNorm':<30} {unfused_rms:>12.1f} {fused_rms:>12.1f} {unfused_rms/fused_rms:>7.2f}x")
        print(f"  {'Residual + RMSNorm':<30} {unfused_res:>12.1f} {fused_res:>12.1f} {unfused_res/fused_res:>7.2f}x")
        print(f"  {'SiLU-Mul':<30} {unfused_silu:>12.1f} {fused_silu:>12.1f} {unfused_silu/fused_silu:>7.2f}x")
        
        # Bandwidth analysis
        bytes_total = batch_size * hidden_size * 4  # float32
        print(f"\n  Effective bandwidth (Fused RMSNorm): "
              f"{2 * bytes_total / (fused_rms * 1e-6) / 1e9:.1f} GB/s "
              f"(read input + write output)")

## 6. CUDA Implementation (for Comparison)

Here's how the same fused RMSNorm would look in CUDA, for comparison with the Triton version.

In [ ]:
# CUDA version of fused RMSNorm (shown as code, compiled with load_inline)

cuda_rmsnorm_source = r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

// Warp-level reduction helper
__device__ float warp_reduce_sum(float val) {
    for (int offset = 16; offset > 0; offset >>= 1) {
        val += __shfl_down_sync(0xFFFFFFFF, val, offset);
    }
    return val;
}

// Block-level reduction using shared memory
__device__ float block_reduce_sum(float val, float* smem) {
    int warp_id = threadIdx.x / 32;
    int lane_id = threadIdx.x % 32;
    int num_warps = blockDim.x / 32;
    
    // Warp-level reduction
    val = warp_reduce_sum(val);
    
    // Store warp results
    if (lane_id == 0) smem[warp_id] = val;
    __syncthreads();
    
    // Final reduction in first warp
    val = (lane_id < num_warps) ? smem[lane_id] : 0.0f;
    val = warp_reduce_sum(val);
    
    return val;  // Only thread 0 has the correct result
}

__global__ void rmsnorm_kernel(
    const float* __restrict__ input,    // (num_tokens, hidden_size)
    const float* __restrict__ weight,   // (hidden_size,)
    float* __restrict__ output,          // (num_tokens, hidden_size)
    int hidden_size,
    float eps
) {
    // Each block handles one row (token)
    int row = blockIdx.x;
    const float* row_input = input + row * hidden_size;
    float* row_output = output + row * hidden_size;
    
    extern __shared__ float smem[];
    
    // Step 1: Compute sum of squares using grid-stride loop
    float sum_sq = 0.0f;
    for (int i = threadIdx.x; i < hidden_size; i += blockDim.x) {
        float val = row_input[i];
        sum_sq += val * val;
    }
    
    // Step 2: Reduce across threads
    sum_sq = block_reduce_sum(sum_sq, smem);
    
    // Step 3: Broadcast RMS to all threads
    __shared__ float shared_rrms;
    if (threadIdx.x == 0) {
        shared_rrms = rsqrtf(sum_sq / hidden_size + eps);
    }
    __syncthreads();
    float rrms = shared_rrms;
    
    // Step 4: Normalize and write output
    for (int i = threadIdx.x; i < hidden_size; i += blockDim.x) {
        row_output[i] = row_input[i] * rrms * weight[i];
    }
}
'''

print("CUDA RMSNorm kernel features:")
print("  - Grid-stride loop: handles arbitrary hidden_size")
print("  - Warp shuffle reduction: efficient sum of squares")
print("  - Block-level reduction: combines warp results via shared memory")
print("  - Shared broadcast: share rrms across all threads")
print()
print(f"Lines of code: CUDA ~{cuda_rmsnorm_source.count(chr(10))} vs Triton ~25")
print("The Triton version achieves similar performance with much less code.")

## 7. Advanced Fusion: Full FFN Block

In an ideal world, we'd fuse the entire FFN block into a single kernel:

```python
# LLaMA FFN block
hidden = residual + attn_output                    # Residual
normed = RMSNorm(hidden)                           # Normalize
gate = W_gate @ normed                             # Gate projection
up = W_up @ normed                                 # Up projection  
activated = SiLU(gate) * up                        # Activation
down = W_down @ activated                          # Down projection
output = hidden + down                             # Residual
```

In practice, we fuse the element-wise operations but not the matmuls (which are compute-bound and handled by cuBLAS/CUTLASS):

- **Fused kernel 1**: `normed, hidden = residual_rmsnorm(attn_output, residual)`
- **cuBLAS**: `gate, up = [W_gate; W_up] @ normed`
- **Fused kernel 2**: `activated = silu_mul(gate, up)`
- **cuBLAS**: `down = W_down @ activated`
- **Fused kernel 3**: `output, residual = residual_rmsnorm(down, hidden)` (for next layer)

In [ ]:
# Simulate the full FFN pipeline with fusion points
if torch.cuda.is_available():
    batch, hidden_size, ffn_size = 256, 4096, 11008
    
    # Create all the tensors
    residual = torch.randn(batch, hidden_size, device='cuda')
    attn_output = torch.randn(batch, hidden_size, device='cuda')
    weight_norm = torch.ones(hidden_size, device='cuda')
    W_gate = torch.randn(ffn_size, hidden_size, device='cuda') * 0.01
    W_up = torch.randn(ffn_size, hidden_size, device='cuda') * 0.01
    W_down = torch.randn(hidden_size, ffn_size, device='cuda') * 0.01
    
    # ---- Fused pipeline ----
    
    # Step 1: Fused residual + RMSNorm
    normed, hidden = fused_residual_rmsnorm(attn_output, residual, weight_norm)
    
    # Step 2: Gate and Up projections (cuBLAS)
    gate_out = torch.mm(normed, W_gate.T)
    up_out = torch.mm(normed, W_up.T)
    
    # Step 3: Fused SiLU-Mul
    activated = fused_silu_mul(gate_out, up_out)
    
    # Step 4: Down projection (cuBLAS)
    down = torch.mm(activated, W_down.T)
    
    print(f"FFN Pipeline Shapes:")
    print(f"  Input:    {attn_output.shape} + {residual.shape}")
    print(f"  Normed:   {normed.shape}")
    print(f"  Gate/Up:  {gate_out.shape}")
    print(f"  Activated:{activated.shape}")
    print(f"  Output:   {down.shape}")
    print()
    print(f"Fusion points:")
    print(f"  1. residual + RMSNorm (saves {batch*hidden_size*4/1e6:.1f} MB IO)")
    print(f"  2. SiLU + multiply   (saves {batch*ffn_size*4/1e6:.1f} MB IO)")

## 8. Summary

| Fused Kernel | Operations Combined | IO Savings | Where Used |
|-------------|--------------------|-----------|-----------|
| **Residual + RMSNorm** | Add + normalize | ~40% | Every transformer layer |
| **SiLU-Mul** | Activation + gating | ~33% | FFN in LLaMA/Mistral |
| **RoPE** | Position rotation | Eliminates intermediate | Attention QK projection |
| **LayerNorm + Linear** | Normalize + project | ~30% | Some model architectures |

### Key Principles

1. **Fuse memory-bound operations** — they share the same data
2. **Don't fuse compute-bound operations** (like matmul) — they already saturate compute
3. **Compute in higher precision** (FP32) in registers, store in lower precision (FP16/BF16)
4. **One program per row** for normalization — natural parallelism over batch/sequence

## Exercises

### Exercise 1: Fused GELU-Mul

Implement a fused GELU activation with gating (used in GPT-style models with GLU variants).

### Exercise 2: Fused RoPE + Attention Score

Fuse RoPE with the Q·K attention score computation — apply RoPE to Q and K inline during dot product.

### Exercise 3: Fused LayerNorm

Implement LayerNorm (with mean subtraction, unlike RMSNorm) in Triton. Compare performance with RMSNorm.

In [ ]:
# Exercise 1 Skeleton
print("Exercise 1: Fused GELU-Mul")
print()
print("GELU(x) = 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))")
print()
print("Fused: output = GELU(gate) * up")
print("Modify the fused_silu_mul_kernel to use GELU instead of SiLU.")
print("Test against: torch.nn.functional.gelu(gate) * up")

In [ ]:
# Exercise 2 Skeleton
print("Exercise 2: Fused RoPE + Attention Score")
print()
print("Instead of:")
print("  q_rot = apply_rope(q, cos, sin, pos)")
print("  k_rot = apply_rope(k, cos, sin, pos)")
print("  score = dot(q_rot, k_rot)")
print()
print("Fuse into:")
print("  score = dot(rope(q), rope(k))")
print("  → Apply rotation inline during dot product computation")
print("  → Never materialize rotated Q or K")

In [ ]:
# Exercise 3 Skeleton

@triton.jit
def layernorm_kernel(
    input_ptr, gamma_ptr, beta_ptr, output_ptr,
    hidden_size, eps,
    stride_in, stride_out,
    BLOCK_SIZE: tl.constexpr,
):
    row_idx = tl.program_id(0)
    col_offsets = tl.arange(0, BLOCK_SIZE)
    mask = col_offsets < hidden_size
    
    # TODO:
    # 1. Load the row
    # 2. Compute mean = sum(x) / hidden_size
    # 3. Compute var = sum((x - mean)^2) / hidden_size
    # 4. Normalize: y = (x - mean) / sqrt(var + eps)
    # 5. Apply gamma and beta: output = y * gamma + beta
    # 6. Store result
    pass

print("Exercise 3: Implement LayerNorm.")
print("Note: LayerNorm requires TWO reductions (mean and variance).")
print("Compare the number of operations with RMSNorm (ONE reduction).")